In [1]:
import os
import requests
import time
import pymol
from Bio.PDB import PDBParser
from tqdm import tqdm
from Bio.PDB import PDBParser, PDBIO, Select

In [6]:
def search_foldseek_api(query_pdb):
    """使用 Foldseek API 寻找结构同源物 (增强健壮性版)"""
    print(f"🚀 [1/4] 正在 Foldseek 搜索同源结构: {os.path.basename(query_pdb)} ...")
    
    if not os.path.exists(query_pdb):
        print(f"❌ 文件不存在: {query_pdb}")
        return []

    with open(query_pdb, 'rb') as f:
        data = f.read()
    
    # === 修改开始: 增加重试机制和详细报错 ===
    max_retries = 3
    for attempt in range(max_retries):
        try:
            req = requests.post(
                'https://search.foldseek.com/api/ticket',
                files={'q': ('query.pdb', data)},
                data={'mode': '3di', 'database[]': ['pdb100']},
                timeout=30  # 增加超时设置
            )

            # 1. 检查 HTTP 状态码
            if req.status_code != 200:
                print(f"⚠️ API 请求失败 (HTTP {req.status_code}): {req.text[:100]}")
                # 如果是限流 (429) 或服务器错误 (5xx)，等待后重试
                if req.status_code in [429, 500, 502, 503, 504]:
                    time.sleep(5 * (attempt + 1))
                    continue
                else:
                    return [] # 其他错误直接退出

            # 2. 安全解析 JSON
            resp_json = req.json()
            if 'id' not in resp_json:
                print(f"❌ API 返回了非预期内容 (无 Ticket ID): {resp_json}")
                return []
            
            ticket = resp_json['id']
            break # 成功拿到 ticket，跳出重试循环

        except requests.exceptions.RequestException as e:
            print(f"⚠️ 网络错误 (尝试 {attempt+1}/{max_retries}): {e}")
            time.sleep(3)
        except Exception as e:
            print(f"❌ 未知错误: {e}")
            return []
    else:
        print("❌多次重试失败，跳过此文件。")
        return []
    # === 修改结束 ===

    # ... 下面是原有的轮询代码 (保持不变) ...
    while True:
        try:
            res = requests.get(f'https://search.foldseek.com/api/ticket/{ticket}', timeout=30)
            # 同样建议在这里增加非 200 判断
            if res.status_code != 200:
                print(f"⏳ 轮询状态异常 (HTTP {res.status_code})，重试中...")
                time.sleep(2)
                continue
                
            status = res.json().get('status')
            if status == "COMPLETE": break
            if status == "ERROR":
                print("❌ Foldseek 服务端处理出错。")
                return []
            
            print("   ⏳ Foldseek 计算中...", end="\r")
            time.sleep(2) # 稍微增加轮询间隔，对服务器友好一点
        except Exception as e:
            print(f"❌ 轮询网络错误: {e}")
            return []

    try:
        res = requests.get(f'https://search.foldseek.com/api/result/{ticket}/0', timeout=30)
        json_data = res.json()
        results_list = json_data.get('results', [])
        
        if not results_list: return []
        first_db_result = results_list[0]
        
        if 'alignments' not in first_db_result:
            return []
            
        real_hits = first_db_result['alignments']
        print(f"   ✅ API 返回了 {len(real_hits)} 个同源结构")
        return real_hits[:20]
        
    except Exception as e:
        print(f"❌ 解析结果错误: {e}")
        return []

def extract_target_from_hit(hit):
    """智能解析 Foldseek 返回的 target 字符串"""
    target_val = ""
    if isinstance(hit, dict):
        target_val = hit.get('target', '')
    elif isinstance(hit, list) and len(hit) > 0:
        if isinstance(hit[0], dict):
            target_val = hit[0].get('target', '')
        elif isinstance(hit[0], str):
            target_val = hit[0]
    return target_val

def has_useful_ligand(pdb_file):
    """检查 PDB 是否含有有机配体"""
    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure('tmp', pdb_file)
        ligands = []
        blacklist = ['HOH', 'WAT', 'DOD', 'SO4', 'PO4', 'GOL', 'EDO', 'CL', 'NA', 'MG', 'ZN', 'CA', 'MN', 'K', 'ACT', 'PEG']
        
        for residue in structure.get_residues():
            if residue.id[0].startswith('H_'):
                resname = residue.get_resname().strip()
                if resname not in blacklist:
                    ligands.append(resname)
        return list(set(ligands))
    except Exception:
        return []

def align_and_extract_skid_style(target_file, template_file, output_sdf):
    print(f"🔧 [3/4] 正在进行结构叠合 (PyMOL) ...")
    try:
        pymol.cmd.reinitialize()
        pymol.cmd.load(target_file, "target")
        pymol.cmd.load(template_file, "template")
        pymol.cmd.remove("resn HOH")
        
        stats = pymol.cmd.align("template", "target", cycles=0)
        print(f"   RMSD: {stats[0]:.3f} Å")
        
        pymol.cmd.select("ref_lig", "template and organic")
        if pymol.cmd.count_atoms("ref_lig") == 0:
            return False

        pymol.cmd.save(output_sdf, "ref_lig")
        return True
    except Exception as e:
        print(f"   ❌ PyMOL 操作失败: {e}")
        return False

def main(protein_file, output_dir):
    print("🏁 程序启动...")
    if not os.path.exists(protein_file):
        print(f"❌ 找不到输入文件: {protein_file}")
        return
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    
    hits = search_foldseek_api(protein_file)
    if not hits: return

    found = False
    
    for hit in hits:
        target_name = extract_target_from_hit(hit)
        if not target_name: continue
        
        # # === 关键修改 1: 过滤 AlphaFold 预测结构 ===
        # if target_name.startswith("AF-"):
        #     # AlphaFold 结构没有底物，直接跳过
        #     continue

        # === 关键修改 2: 修复 ID 解析 (处理 '7ES2-ASSEMBLY1' 这种情况) ===
        # 先按 '_' 分割，再按 '-' 分割，最后按 '.' 分割，取第一个部分并截取前4位
        # 例子: "7ES2-ASSEMBLY1" -> "7ES2", "1abc_A" -> "1ABC"
        raw_id = target_name.split('_')[0].split('-')[0].split('.')[0]
        pdb_id = raw_id[:4].upper()
        
        if len(pdb_id) != 4:
            print(f"⚠️ 跳过非法 ID 格式: {target_name} -> {pdb_id}")
            continue

        print(f"\n🔍 分析同源 PDB: {pdb_id} (原始: {target_name}) ...")
        
        # 2. 下载 PDB
        pdb_url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        save_path = os.path.join(output_dir, f"{pdb_id}.pdb")
        
        if not os.path.exists(save_path):
            try:
                r = requests.get(pdb_url, timeout=15)
                if r.status_code == 200:
                    with open(save_path, "wb") as f:
                        f.write(r.content)
                else:
                    print(f"   无法下载 PDB (HTTP {r.status_code})")
                    continue
            except:
                print(f"   下载超时")
                continue
        
        # 3. 检查是否有底物
        ligands = has_useful_ligand(save_path)
        if not ligands:
            print("   ❌ 空结构或仅含离子/水，跳过")
            continue
        
        print(f"   ✅ 发现潜在底物: {ligands}")
        
        # 4. 叠合并提取
        ref_sdf_path = os.path.join(output_dir, "ref_ligand.sdf")
        success = align_and_extract_skid_style(protein_file, save_path, ref_sdf_path)
        
        if success:
            print(f"\n🎉 [4/4] 成功！对接位点参考文件已生成: {ref_sdf_path}")
            print(f"   最佳同源模版: {pdb_id} (底物: {ligands[0]})")
            print(f"   请在 GNINA 中使用: --autobox_ligand \"{ref_sdf_path}\"")
            found = True
            break 
    if not found:
        print("\n❌ 所有的同源结构似乎都没有配体。")
    
    print("\n✅ 所有任务运行结束。")

In [9]:
import random
pdb_folder = [
            # r'F:\反应条件生成对比实验\ZymCTRL\OmegaFold',
            r'F:\反应条件生成对比实验\GENzyme\all_protein',
            r'F:\反应条件生成对比实验\REXzyme\generated\OmegaFold',
            r'F:\反应条件生成对比实验\ProCALM\generated\OmegaFold',
            r'F:\反应条件生成对比实验\TransGen\OmegaFold']
for sub_folder in pdb_folder:
    parent_folder = os.path.dirname(sub_folder)
    parent_folder_foldseek = os.path.join(parent_folder,"foldseek")
    if not os.path.exists(parent_folder_foldseek):
        os.makedirs(parent_folder_foldseek)
    pdb_list = os.listdir(sub_folder)
    if len(pdb_list) >200:
        pdb_list = random.sample(pdb_list, 200)
    for pdb_o in pdb_list:
        protein_file = os.path.join(sub_folder,pdb_o)
        save_path = os.path.join(parent_folder_foldseek,pdb_o)
        main(protein_file, save_path)

🏁 程序启动...
🚀 [1/4] 正在 Foldseek 搜索同源结构: RHEA_57980__protein_5.pdb ...
   ✅ API 返回了 1 个同源结构

🔍 分析同源 PDB: 4IJQ (原始: 4ijq-assembly1.cif.gz_A Human hypoxanthine-guanine phosphoribosyltransferase in complex with [(2-((Guanine-9H-yl)methyl)propane-1,3-diyl)bis(oxy)]bis(methylene))diphosphonic acid) ...
   ✅ 发现潜在底物: ['SV2']
🔧 [3/4] 正在进行结构叠合 (PyMOL) ...
   RMSD: 9.520 Å

🎉 [4/4] 成功！对接位点参考文件已生成: F:\反应条件生成对比实验\GENzyme\foldseek\RHEA_57980__protein_5.pdb\ref_ligand.sdf
   最佳同源模版: 4IJQ (底物: SV2)
   请在 GNINA 中使用: --autobox_ligand "F:\反应条件生成对比实验\GENzyme\foldseek\RHEA_57980__protein_5.pdb\ref_ligand.sdf"

✅ 所有任务运行结束。
🏁 程序启动...
🚀 [1/4] 正在 Foldseek 搜索同源结构: RHEA_57980__protein_4.pdb ...
   ✅ API 返回了 0 个同源结构
🏁 程序启动...
🚀 [1/4] 正在 Foldseek 搜索同源结构: RHEA_81939__protein_1.pdb ...
   ✅ API 返回了 1 个同源结构

🔍 分析同源 PDB: 6Z9L (原始: 6z9l-assembly1.cif.gz_A Enterococcal PrgA) ...
   ❌ 空结构或仅含离子/水，跳过

❌ 所有的同源结构似乎都没有配体。

✅ 所有任务运行结束。
🏁 程序启动...
🚀 [1/4] 正在 Foldseek 搜索同源结构: RHEA_51504__protein_2.pdb ...
   ✅ API 返回了 0 个同源结构
🏁 程序启

PermissionError: [Errno 13] Permission denied: 'F:\\反应条件生成对比实验\\TransGen\\OmegaFold\\plots'

In [47]:
# ================= 用户配置区 =================
protein_file = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\ESMFold_for_experments\N178Y_I146F_Y164G_ESMFold.pdb"
output_dir = r"E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\N178Y_I146F_Y164G_FoldSeek"
# ===========================================

In [48]:
pdb_file = r'E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\Uniprot_AF2'
save_file = r'E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\Uniprot_AF2_Foldseek\PDB'

In [49]:
pdb_list = os.listdir(pdb_file)

In [50]:
for pdb_s in tqdm(pdb_list):
    save_name = pdb_s.split("_")[0]
    save_path = os.path.join(save_file,save_name)
    protein_file = os.path.join(pdb_file,pdb_s)
    main(protein_file, save_path)
    # break

  0%|          | 0/4908 [00:00<?, ?it/s]

🏁 程序启动...
🚀 [1/4] 正在 Foldseek 搜索同源结构: A0A022PPJ0_pLDDT91.4.pdb ...


  0%|          | 1/4908 [00:03<5:09:18,  3.78s/it]

   ✅ API 返回了 1 个同源结构

🔍 分析同源 PDB: 6L90 (原始: 6l90-assembly1.cif.gz_A Crystal structure of ugt transferase enzyme) ...
   ❌ 空结构或仅含离子/水，跳过

❌ 所有的同源结构似乎都没有配体。

✅ 所有任务运行结束。
🏁 程序启动...
🚀 [1/4] 正在 Foldseek 搜索同源结构: A0A022PPJ8_pLDDT94.0.pdb ...


  0%|          | 2/4908 [00:07<4:57:08,  3.63s/it]

...

❌多次重试失败，跳过此文件。
🏁 程序启动...
🚀 [1/4] 正在 Foldseek 搜索同源结构: W8QNN5_pLDDT88.5.pdb ...
⚠️ API 请求失败 (HTTP 429): {"status":"RATELIMIT","reason":"The Foldseek server is a shared resource. If you submit a lot of job
⚠️ API 请求失败 (HTTP 429): {"status":"RATELIMIT","reason":"The Foldseek server is a shared resource. If you submit a lot of job
   ✅ API 返回了 1 个同源结构

🔍 分析同源 PDB: 8INH (原始: 8inh-assembly2.cif.gz_A ZjOGT3, flavonoid 7,4'-di-O-glycosyltransferase) ...


100%|█████████▉| 4907/4908 [36:29:18<00:31, 31.37s/it]

   ✅ 发现潜在底物: ['UDP']
🔧 [3/4] 正在进行结构叠合 (PyMOL) ...
   RMSD: 7.722 Å

🎉 [4/4] 成功！对接位点参考文件已生成: E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\Uniprot_AF2_Foldseek\PDB\W8QNN5\ref_ligand.sdf
   最佳同源模版: 8INH (底物: UDP)
   请在 GNINA 中使用: --autobox_ligand "E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\Uniprot_AF2_Foldseek\PDB\W8QNN5\ref_ligand.sdf"

✅ 所有任务运行结束。
🏁 程序启动...
🚀 [1/4] 正在 Foldseek 搜索同源结构: X2D877_pLDDT94.5.pdb ...
⚠️ API 请求失败 (HTTP 429): {"status":"RATELIMIT","reason":"The Foldseek server is a shared resource. If you submit a lot of job
⚠️ API 请求失败 (HTTP 429): {"status":"RATELIMIT","reason":"The Foldseek server is a shared resource. If you submit a lot of job
⚠️ API 请求失败 (HTTP 429): {"status":"RATELIMIT","reason":"The Foldseek server is a shared resource. If you submit a lot of job


100%|██████████| 4908/4908 [36:29:51<00:00, 26.77s/it]

❌多次重试失败，跳过此文件。


In [37]:
# 把PDB结构中的A链提取出来，放入另外一个文件夹

In [38]:
pp_path = r'E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\Uniprot_AF2_Foldseek\PDB'
pp_save = r'E:\pUGTdb\20251128_collection\download_PF00201_taxonomy33090\step2_get_3D_Protein\Uniprot_AF2_Foldseek\single_chain_PDB'
pp_file = os.listdir(pp_path)

In [39]:
class ChainSelect(Select):
    def accept_chain(self, chain):
        return chain.id == "A"   # 改成你要的链
for pp in pp_file:
    p_path = os.path.join(pp_path,pp)
    sub_pp = os.listdir(p_path)
    for p in sub_pp:
        if p.endswith('pdb'):
            p_ = os.path.join(p_path, p)
            p_s=  os.path.join(pp_save, p)
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure("prot", p_)
            io = PDBIO()
            io.set_structure(structure)
            io.save(p_s, ChainSelect())